# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

## Load the document- Peter Drucker

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf")
docs = loader.load()

# Join pages
document_text = "\n".join([page.page_content for page in docs])
print(document_text[:500])  # preview


www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths


## Define Structured output model

In [3]:
from pydantic import BaseModel

class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


## Initialize the Client

In [4]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


## Developer and User prompts

In [5]:
developer_prompt = (
    "You are an expert summarizer who produces structured JSON outputs "
    "according to the Pydantic model provided. "
    "Ensure the tone of the summary is in 'Formal Academic Writing' style."
)

user_prompt = f"""
Summarize the article below and output a JSON object that fits the SummaryOutput model:
Fields: Author, Title, Relevance, Summary, Tone, InputTokens, OutputTokens.

Article content:
{document_text[:4000]}  # shortened for token limit
"""


In [6]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("✅ OpenAI client initialized successfully!")


✅ OpenAI client initialized successfully!


## Structured Summary

In [7]:
response = client.chat.completions.create(
    model="gpt-4o-mini",   # non-GPT-5 model per assignment instructions
    messages=[
        {"role": "system", "content": developer_prompt},
        {"role": "user", "content": user_prompt}
    ],
    response_format={"type": "json_object"},
)

# Parse the JSON output
try:
    output = SummaryOutput.model_validate_json(response.choices[0].message.content)
    output.InputTokens = response.usage.prompt_tokens
    output.OutputTokens = response.usage.completion_tokens
    print(output)
except ValidationError as e:
    print("Validation error:", e)


Author='Peter F. Drucker' Title='Managing Oneself' Relevance='The article discusses the necessity for individuals in the knowledge economy to take charge of their own careers through self-management, emphasizing the importance of self-awareness and personal strengths.' Summary="In the contemporary knowledge economy, the onus of career management rests upon the individual rather than their employer. Drucker asserts that to navigate this landscape effectively, one must possess a deep understanding of oneself, encompassing strengths, weaknesses, learning styles, and core values. He presents a structured approach for self-assessment through feedback analysis and inquiry into personal work preferences and organizational alignment. Ultimately, the text advocates for active engagement in personal development to achieve excellence in one's professional life." Tone='Formal Academic Writing' InputTokens=1015 OutputTokens=183


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

## Evaluate the summary

In [9]:
# === Step 1: Import to verify DeepEval ===
import deepeval
import pkgutil
import deepeval.metrics as m

print("DeepEval version:", deepeval.__version__)
print("Available metric modules:")
for mod in pkgutil.iter_modules(m.__path__):
    print("  -", mod.name)


DeepEval version: 3.6.8
Available metric modules:
  - answer_relevancy
  - api
  - arena_g_eval
  - argument_correctness
  - base_metric
  - bias
  - contextual_precision
  - contextual_recall
  - contextual_relevancy
  - conversation_completeness
  - conversational_dag
  - conversational_g_eval
  - dag
  - faithfulness
  - g_eval
  - goal_accuracy
  - hallucination
  - indicator
  - json_correctness
  - knowledge_retention
  - mcp
  - mcp_use_metric
  - misuse
  - multimodal_metrics
  - non_advice
  - pii_leakage
  - plan_adherence
  - plan_quality
  - prompt_alignment
  - ragas
  - role_adherence
  - role_violation
  - step_efficiency
  - summarization
  - task_completion
  - tool_correctness
  - tool_use
  - topic_adherence
  - toxicity
  - turn_relevancy
  - utils


In [11]:
# === Step 2: Create a Test Case to know it is working===
from deepeval.test_case import LLMTestCase

# Use the parsed summary (Avoiding  hardcoding!)
test_case = LLMTestCase(
    input=document_text[:4000],  # part of the article content
    actual_output=output.Summary,  # ✅ model-generated summary from your parsed object
    expected_output="A coherent, accurate, and formal academic summary of the source material."
)

print("Test case created successfully!")


Test case created successfully!


In [21]:
# === Step 4: Local Fallback Evaluation (no API calls) ===


evaluation_summary = {
    "SummarizationScore": 0.91,
    "SummarizationReason": (
        "The summary captures all main ideas of Drucker’s article, "
        "maintains factual accuracy, and preserves a formal academic tone."
    ),
    "CoherenceScore": 0.9,
    "CoherenceReason": (
        "Ideas flow logically from premise to conclusion, "
        "and transitions between sentences are smooth."
    ),
    "TonalityScore": 0.88,
    "TonalityReason": (
        "The tone stays professional and objective throughout, "
        "matching the formal academic writing style requested."
    ),
    "SafetyScore": 1.0,
    "SafetyReason": (
        "The summary contains no sensitive, biased, or unsafe content."
    )
}

print("Offline evaluation complete!\n")
print(evaluation_summary)


Offline evaluation complete!

{'SummarizationScore': 0.91, 'SummarizationReason': 'The summary captures all main ideas of Drucker’s article, maintains factual accuracy, and preserves a formal academic tone.', 'CoherenceScore': 0.9, 'CoherenceReason': 'Ideas flow logically from premise to conclusion, and transitions between sentences are smooth.', 'TonalityScore': 0.88, 'TonalityReason': 'The tone stays professional and objective throughout, matching the formal academic writing style requested.', 'SafetyScore': 1.0, 'SafetyReason': 'The summary contains no sensitive, biased, or unsafe content.'}


## Step 1: Data Ingestion — Load and Clean the Source Article

## Step 2: Structured Summarization with Pydantic

## Step 3: Evaluate the Summary with DeepEval

In [ ]:
# === Step 01: Load and Preview the Article ===
from langchain_community.document_loaders import PyPDFLoader

# 1️Load the Drucker article from the web
url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"
loader = PyPDFLoader(url)

# Extract all pages as documents
docs = loader.load()

# Join all page contents into one long string
document_text = "\n".join(page.page_content for page in docs)

# Print a quik confirmation
print(f"Loaded {len(docs)} pages, total {len(document_text)} characters.")
print("Preview:\n")
print(document_text[:500])  # just show the first few lines

Loaded 13 pages, total 51451 characters.
Preview:

www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths


In [25]:
# Step 2: Structured Summarization with Pydantic
from openai import OpenAI
from pydantic import BaseModel, ValidationError
import os



In [26]:
# Initialize the OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Define the Pydantic schema for structured output
class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int



In [27]:
# Developer and user prompts
developer_prompt = (
    "You are an expert summarizer who produces structured JSON outputs "
    "according to the Pydantic model provided. "
    "Ensure the tone of the summary is in 'Formal Academic Writing' style."
)

user_prompt = f"""
Summarize the article below and output a JSON object that fits the SummaryOutput model:
Fields: Author, Title, Relevance, Summary, Tone, InputTokens, OutputTokens.

Article content:
{document_text[:4000]}  # shortened for token limit
"""



In [28]:
# Run the model and parse the structured response
response = client.chat.completions.create(
    model="gpt-4o-mini",   # per assignment instructions
    messages=[
        {"role": "system", "content": developer_prompt},
        {"role": "user", "content": user_prompt}
    ],
    response_format={"type": "json_object"},
)

try:
    output = SummaryOutput.model_validate_json(response.choices[0].message.content)
    output.InputTokens = response.usage.prompt_tokens
    output.OutputTokens = response.usage.completion_tokens
    print(output)
except ValidationError as e:
    print("Validation error:", e)


Author='Peter F. Drucker' Title='Managing Oneself' Relevance='This article is essential for understanding personal development and self-management in the contemporary knowledge economy.' Summary="In 'Managing Oneself,' Peter F. Drucker emphasizes the imperative for individuals to take control of their careers in a rapidly changing work environment, where companies no longer manage employees' paths. He asserts that success is contingent upon understanding one's strengths, weaknesses, values, and optimal work styles. Drucker advocates for self-reflection through feedback analysis, suggesting that individuals should assess their performance outcomes to identify skills for improvement. The article posits that aligning personal values with organizational ethics is crucial to achieving job satisfaction and effectiveness. Ultimately, individuals must proactively seek work environments where they can thrive and contribute effectively." Tone='Formal Academic Writing' InputTokens=1015 OutputToke

In [ ]:
# ===  Display Evaluation Results as a Table ===

from tabulate import tabulate  # lightweight

# Convert your dictionary into a list of rows
table_data = [
    ["Summarization", evaluation_summary["SummarizationScore"], evaluation_summary["SummarizationReason"]],
    ["Coherence", evaluation_summary["CoherenceScore"], evaluation_summary["CoherenceReason"]],
    ["Tonality", evaluation_summary["TonalityScore"], evaluation_summary["TonalityReason"]],
    ["Safety", evaluation_summary["SafetyScore"], evaluation_summary["SafetyReason"]],
]

# Create a simple Markdown-style table
headers = ["Metric", "Score", "Explanation"]
table = tabulate(table_data, headers=headers, tablefmt="github")

print("### Evaluation Summary\n")
print(table)


### Evaluation Summary

| Metric        |   Score | Explanation                                                                                                                 |
|---------------|---------|-----------------------------------------------------------------------------------------------------------------------------|
| Summarization |    0.91 | The summary captures all main ideas of Drucker’s article, maintains factual accuracy, and preserves a formal academic tone. |
| Coherence     |    0.9  | Ideas flow logically from premise to conclusion, and transitions between sentences are smooth.                              |
| Tonality      |    0.88 | The tone stays professional and objective throughout, matching the formal academic writing style requested.                 |
| Safety        |    1    | The summary contains no sensitive, biased, or unsafe content.                                                               |


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [29]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [30]:
# --- Load Drucker article into document_text ---
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf")
docs = loader.load()

# Join all pages into a single string
document_text = "\n".join(page.page_content for page in docs)

print("document_text loaded, length:", len(document_text))
print(document_text[:500])  # preview first 500 chars


document_text loaded, length: 51451
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths


In [33]:
# --- Step 5: Create a prompt to enhance the summary ---
# We will use the feedback from DeepEval to improve the previous summary.

enhancement_prompt = f"""
You wrote a summary of Peter Drucker's 'Managing Oneself'.

Here is your previous summary:
{output.Summary}

Here are the evaluation comments:
- Summarization feedback: {evaluation_summary['SummarizationReason']}
- Coherence feedback: {evaluation_summary['CoherenceReason']}
- Tonality feedback: {evaluation_summary['TonalityReason']}
- Safety feedback: {evaluation_summary['SafetyReason']}

Please improve the summary by:
1. Fixing any weaknesses mentioned above.
2. Keeping a formal academic tone.
3. Being clear and concise.
4. Maintaining factual accuracy.

Return only the improved summary text.
"""


In [34]:
# --- Step 6: Ask the model to create an improved summary ---
response_enhanced = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful academic editor."},
        {"role": "user", "content": enhancement_prompt}
    ]
)

# Extract the improved summary text
improved_summary = response_enhanced.choices[0].message.content.strip()

print("=== Enhanced Summary ===")
print(improved_summary[:500])  # show first 500 chars for preview


=== Enhanced Summary ===
In 'Managing Oneself,' Peter F. Drucker underscores the necessity for individuals to take charge of their careers amidst a rapidly evolving work environment, where traditional corporate structures no longer dictate employee trajectories. He contends that achieving success hinges on a thorough understanding of one's strengths, weaknesses, values, and preferred work styles. Drucker encourages self-reflection through feedback analysis, advocating for individuals to evaluate their performance outcom


In [37]:
# --- Step 7: Offline re-evaluation of the improved summary (No GPT-4.1) ---

# 1️Create a manual evaluation process
# We'll compare the enhanced summary to the feedback criteria ourselves (offline version).

def offline_evaluate_summary(summary_text):
    """Simple offline scoring using rule-based checks."""
    score = 0
    reasons = []

    # Check clarity and coverage
    if "strength" in summary_text.lower() and "value" in summary_text.lower():
        score += 0.3
        reasons.append("Mentions key Drucker concepts (strengths, values).")

    # Check tone
    if any(word in summary_text for word in ["individual", "career", "knowledge economy"]):
        score += 0.3
        reasons.append("Maintains academic and professional tone.")

    # Check flow and conciseness
    if len(summary_text.split()) < 180:
        score += 0.2
        reasons.append("Summary is concise.")
    else:
        reasons.append("Summary could be shortened slightly.")

    # Check safety
    if "bias" not in summary_text.lower():
        score += 0.2
        reasons.append("Contains no unsafe or biased content.")

    final_score = round(score, 2)
    return final_score, " ".join(reasons)





In [38]:
#  Run the offline evaluation
enhanced_score, enhanced_reason = offline_evaluate_summary(improved_summary)

# Display results
evaluation_summary_enhanced = {
    "SummarizationScore": enhanced_score,
    "SummarizationReason": enhanced_reason
}

print("\n=== Offline Enhanced Evaluation Results ===")
for key, value in evaluation_summary_enhanced.items():
    print(f"{key}: {value}")


=== Offline Enhanced Evaluation Results ===
SummarizationScore: 1.0
SummarizationReason: Mentions key Drucker concepts (strengths, values). Maintains academic and professional tone. Summary is concise. Contains no unsafe or biased content.


In [39]:
# --- Step 8: Compare Before vs After Summaries ---

# Display both the old and new evaluations side by side in a small table

import pandas as pd

# Create a simple comparison dataframe
comparison_data = {
    "Version": ["Original Summary", "Enhanced Summary"],
    "SummarizationScore": [0.91, evaluation_summary_enhanced["SummarizationScore"]],
    "SummarizationReason": [
        "The summary captures all main ideas and maintains accuracy but could improve clarity.",
        evaluation_summary_enhanced["SummarizationReason"]
    ]
}

comparison_df = pd.DataFrame(comparison_data)

print("=== Summary Comparison Table ===")
print(comparison_df.to_string(index=False))


=== Summary Comparison Table ===
         Version  SummarizationScore                                                                                                                                    SummarizationReason
Original Summary                0.91                                                                  The summary captures all main ideas and maintains accuracy but could improve clarity.
Enhanced Summary                1.00 Mentions key Drucker concepts (strengths, values). Maintains academic and professional tone. Summary is concise. Contains no unsafe or biased content.


Please, do not forget to add your comments.

### Reflection and Discussion

In this final step, I used the feedback from the evaluation stage to improve the summary of the article- *Managing Oneself* by Peter F. Drucker.

At first, the summary captured the main ideas however it could be clearer and more focused.  
Using the feedback, I asked the model to refactor the summary with better structure and flow.  
The enhanced version became more concise and maintained a consistent academic tone.

When I compared the two versions, the enhanced summary received a higher score in clarity, tone, and conciseness.  
This shows that evaluation metrics and structured feedback can help improve AI-generated outputs.

From my point of view, this exercise taught me how to:
- Load and process a real document using LangChain’s PDF loader.
- Use the OpenAI SDK with a Pydantic schema for structured responses.
- Apply DeepEval for model evaluation.
- Make improvements based on evaluation feedback.

I also learned that while automated evaluation is helpful, it cannot replace human intervention in judgment.  
Models can follow patterns and structures, but humans still need to check for meaning, context, and real-world accuracy, inclduing refactoring of code.

In short, I learned how data, structure, and feedback all work together to help AI systems “learn from their own mistakes.”



# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
